In [1]:
# Step 1: Import necessary libraries
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding
from tensorflow.keras.models import Sequential

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.18.1


In [6]:
# Our dummy sentences
dummy_sentences = [
    "I love machine learning",
    "I love deep learning",
    "machine learning is fun",
    "deep learning is amazing",
    "I love fun"
]

print("Our dummy sentences:")
for i, sent in enumerate(dummy_sentences):
    print(f"  {i+1}. {sent}")

Our dummy sentences:
  1. I love machine learning
  2. I love deep learning
  3. machine learning is fun
  4. deep learning is amazing
  5. I love fun


In [7]:
# create a vocabulary. 

# Create a tokenizer and fit on our sentences
tokenizer = Tokenizer()
tokenizer.fit_on_texts(dummy_sentences)


# Get the word index (vocabulary)
word_index = tokenizer.word_index
vocab_size = len(word_index) + 1  # +1 because index 0 is reserved for padding

print("=" * 60)
print("WORD INDEX (Vocabulary)")
print("=" * 60)
print(f"\nTotal unique words: {len(word_index)}")
print(f"Vocabulary size (with padding): {vocab_size}")
print("\nWord → Index mapping:")
print("-" * 30)
for word, idx in sorted(word_index.items(), key=lambda x: x[1]):
    print(f"  '{word}' → {idx}")

WORD INDEX (Vocabulary)

Total unique words: 8
Vocabulary size (with padding): 9

Word → Index mapping:
------------------------------
  'learning' → 1
  'i' → 2
  'love' → 3
  'machine' → 4
  'deep' → 5
  'is' → 6
  'fun' → 7
  'amazing' → 8


In [8]:
# Convert sentences to sequences of integers
sequences = tokenizer.texts_to_sequences(dummy_sentences)

print("=" * 60)
print("SENTENCES → SEQUENCES")
print("=" * 60)
for sent, seq in zip(dummy_sentences, sequences):
    print(f"\nSentence: '{sent}'")
    print(f"Sequence: {seq}")
    # Show word-by-word mapping
    words = sent.lower().split()
    mapping = " → ".join([f"'{w}':{word_index[w]}" for w in words])
    print(f"Mapping:  {mapping}")

SENTENCES → SEQUENCES

Sentence: 'I love machine learning'
Sequence: [2, 3, 4, 1]
Mapping:  'i':2 → 'love':3 → 'machine':4 → 'learning':1

Sentence: 'I love deep learning'
Sequence: [2, 3, 5, 1]
Mapping:  'i':2 → 'love':3 → 'deep':5 → 'learning':1

Sentence: 'machine learning is fun'
Sequence: [4, 1, 6, 7]
Mapping:  'machine':4 → 'learning':1 → 'is':6 → 'fun':7

Sentence: 'deep learning is amazing'
Sequence: [5, 1, 6, 8]
Mapping:  'deep':5 → 'learning':1 → 'is':6 → 'amazing':8

Sentence: 'I love fun'
Sequence: [2, 3, 7]
Mapping:  'i':2 → 'love':3 → 'fun':7


In [9]:
# Pad sequences to same length
max_length = 5  # We'll pad all to length 5
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post')

print("=" * 60)
print("PADDED SEQUENCES")
print("=" * 60)
print(f"\nMax length set to: {max_length}")
print(f"Padding value: 0 (represents nothing/padding)")
print("\nOriginal → Padded:")
print("-" * 50)
for sent, seq, padded in zip(dummy_sentences, sequences, padded_sequences):
    print(f"\n'{sent}'")
    print(f"  Original: {seq}")
    print(f"  Padded:   {list(padded)}")

PADDED SEQUENCES

Max length set to: 5
Padding value: 0 (represents nothing/padding)

Original → Padded:
--------------------------------------------------

'I love machine learning'
  Original: [2, 3, 4, 1]
  Padded:   [np.int32(2), np.int32(3), np.int32(4), np.int32(1), np.int32(0)]

'I love deep learning'
  Original: [2, 3, 5, 1]
  Padded:   [np.int32(2), np.int32(3), np.int32(5), np.int32(1), np.int32(0)]

'machine learning is fun'
  Original: [4, 1, 6, 7]
  Padded:   [np.int32(4), np.int32(1), np.int32(6), np.int32(7), np.int32(0)]

'deep learning is amazing'
  Original: [5, 1, 6, 8]
  Padded:   [np.int32(5), np.int32(1), np.int32(6), np.int32(8), np.int32(0)]

'I love fun'
  Original: [2, 3, 7]
  Padded:   [np.int32(2), np.int32(3), np.int32(7), np.int32(0), np.int32(0)]


In [11]:
# Create an Embedding layer
embedding_dim = 4  # Each word will be represented by 4 numbers

# Create a simple model with just an embedding layer
embedding_layer = Embedding(
    input_dim=vocab_size,      # Size of vocabulary (10 words + padding)
    output_dim=embedding_dim,   # Dimension of embeddings (4-dimensional vectors)
    input_length=max_length     # Length of input sequences (5)
)

print("=" * 60)
print("EMBEDDING LAYER CONFIGURATION")
print("=" * 60)
print(f"\n• Vocabulary size (input_dim): {vocab_size}")
print(f"• Embedding dimension (output_dim): {embedding_dim}")
print(f"• Input sequence length: {max_length}")
print(f"\n• Embedding matrix shape: ({vocab_size}, {embedding_dim})")
print(f"  → {vocab_size} words × {embedding_dim} dimensions")
print(f"  → Total trainable parameters: {vocab_size * embedding_dim}")

EMBEDDING LAYER CONFIGURATION

• Vocabulary size (input_dim): 9
• Embedding dimension (output_dim): 4
• Input sequence length: 5

• Embedding matrix shape: (9, 4)
  → 9 words × 4 dimensions
  → Total trainable parameters: 36


In [12]:
# Build the layer by passing input
_ = embedding_layer(padded_sequences)

# Get the embedding matrix (weights)
embedding_matrix = embedding_layer.get_weights()[0]

print("=" * 60)
print("EMBEDDING MATRIX (Random Initialization)")
print("=" * 60)
print(f"\nShape: {embedding_matrix.shape}")
print(f"→ {embedding_matrix.shape[0]} rows (one per word index)")
print(f"→ {embedding_matrix.shape[1]} columns (embedding dimensions)")

print("\n" + "-" * 60)
print("Index | Word        | Embedding Vector (4 dimensions)")
print("-" * 60)

# Create reverse mapping: index → word
index_to_word = {v: k for k, v in word_index.items()}
index_to_word[0] = "<PAD>"  # Index 0 is padding

for idx in range(vocab_size):
    word = index_to_word.get(idx, "<UNK>")
    vector = embedding_matrix[idx]
    vector_str = "[" + ", ".join([f"{v:+.4f}" for v in vector]) + "]"
    print(f"  {idx:2d}  | {word:11s} | {vector_str}")

EMBEDDING MATRIX (Random Initialization)

Shape: (9, 4)
→ 9 rows (one per word index)
→ 4 columns (embedding dimensions)

------------------------------------------------------------
Index | Word        | Embedding Vector (4 dimensions)
------------------------------------------------------------
   0  | <PAD>       | [+0.0306, +0.0432, +0.0117, +0.0105]
   1  | learning    | [-0.0393, +0.0268, -0.0042, +0.0293]
   2  | i           | [+0.0433, +0.0428, +0.0297, -0.0035]
   3  | love        | [+0.0235, +0.0166, -0.0048, +0.0321]
   4  | machine     | [-0.0389, +0.0066, +0.0143, +0.0257]
   5  | deep        | [-0.0491, -0.0284, -0.0229, -0.0335]
   6  | is          | [-0.0003, +0.0165, +0.0239, -0.0269]
   7  | fun         | [+0.0042, -0.0179, -0.0151, +0.0437]
   8  | amazing     | [-0.0122, +0.0170, +0.0186, +0.0255]


2026-05-07 21:44:25.063864: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3 Max
2026-05-07 21:44:25.063901: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 128.00 GB
2026-05-07 21:44:25.063906: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 53.76 GB
I0000 00:00:1778170465.063925 65439044 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1778170465.063948 65439044 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
